# Pipeline Hyperparameter Tuning (Parallel)

This notebook performs hyperparameter tuning on the **entire pipeline** using **parallel processing**.
It iterates through different parameter combinations, runs the full pipeline for specified test periods in parallel, and saves all results.

**Test Periods:**
- 2025-09 (Auction) / 2025-09 (Market)
- 2025-10 (Auction) / 2025-10 (Market)

**Output Structure:**
- `/opt/temp/haoyan/param_search/`
    - `metrics/`: Parquet files containing parameters and key metrics.
    - `per_outage/`: Parquet files containing per-outage prediction results.
    - `agg/`: Parquet files containing monthly aggregated prediction results.

In [1]:
from pbase.config.ray import init_ray
import shadow_price_prediction

# Initialize Ray for parallel processing
extra_modules = [shadow_price_prediction]
init_ray(address='ray://10.8.0.36:10001', extra_modules=extra_modules)

import pandas as pd
import numpy as np
import random
from pathlib import Path
from pbase.utils.ray import parallel_equal_pool

# Import tuning utilities for resumable search
from shadow_price_prediction.tuning_utils import (
    load_previous_params,
    sample_params,
    run_single_experiment
)

pd.set_option('display.max_columns', 99)

2025-11-19 19:45:33,994	INFO client_builder.py:242 -- Passing the following kwargs to ray.init() on the server: log_to_driver


In [2]:
N_ITER = 10000  # Total number of experiments to run (including previous ones)
BASE_OUTPUT_DIR = Path('/opt/temp/haoyan/param_search')

## 5. Analyze Best Results

In [6]:
metrics_dir = BASE_OUTPUT_DIR / 'metrics'

if metrics_dir.exists():
    all_metrics = []
    for file in metrics_dir.glob('iter_*.parquet'):
        try:
            df = pd.read_parquet(file)
            all_metrics.append(df)
        except Exception as e:
            print(f"Warning: Could not load {file.name}: {e}")
    
    if all_metrics:
        combined_metrics = pd.concat(all_metrics, ignore_index=True)
        
        print(f"Total runs analyzed: {len(combined_metrics)}")
        
        # Best by F1 Score
        if 'F1' in combined_metrics.columns:
            best_idx = combined_metrics['F1'].idxmax()
            best_result = combined_metrics.iloc[best_idx]
            
            print("\n" + "=" * 60)
            print("Best Run (by F1 Score)")
            print("=" * 60)
            print(f"Iteration: {best_result.get('iteration', 'N/A')}")
            print(f"F1 Score: {best_result['F1']:.4f}")
            if 'MAE' in best_result:
                print(f"MAE: {best_result['MAE']:.4f}")
            if 'Precision' in best_result:
                print(f"Precision: {best_result['Precision']:.4f}")
            if 'Recall' in best_result:
                print(f"Recall: {best_result['Recall']:.4f}")
            
            # Print parameters
            print("\nParameters:")
            param_cols = [col for col in combined_metrics.columns 
                         if col not in ['F1', 'Precision', 'Recall', 'MAE', 'RMSE', 'R2',
                                       'iteration', 'timestamp', 'run_id', 'metrics_file',
                                       'per_outage_file', 'agg_file']]
            for col in param_cols:
                print(f"  {col}: {best_result[col]}")
        
        # Best by MAE (lower is better)
        if 'MAE' in combined_metrics.columns:
            best_idx = combined_metrics['MAE'].idxmin()
            best_result = combined_metrics.iloc[best_idx]
            
            print("\n" + "=" * 60)
            print("Best Run (by MAE)")
            print("=" * 60)
            print(f"Iteration: {best_result.get('iteration', 'N/A')}")
            print(f"MAE: {best_result['MAE']:.4f}")
            if 'F1' in best_result:
                print(f"F1 Score: {best_result['F1']:.4f}")
            if 'RMSE' in best_result:
                print(f"RMSE: {best_result['RMSE']:.4f}")
            
            # Print parameters
            print("\nParameters:")
            for col in param_cols:
                print(f"  {col}: {best_result[col]}")

Total runs analyzed: 257


## 6. View Results Summary

In [15]:
combined_metrics.groupby('reg_xgb_thres')[['mae', 'rmse', 'mape', 'precision', 'recall', 'f1_score']].describe()

mae                                                            \
              count        mean       std         min         25%         50%   
reg_xgb_thres                                                                   
0.3            76.0  353.708462  6.633924  338.872654  349.105321  353.149600   
0.5            85.0  350.455663  6.508571  339.434233  344.975163  350.132259   
0.7            96.0  351.796770  5.505171  338.131928  347.552162  351.337206   

                                       rmse                          \
                      75%         max count         mean        std   
reg_xgb_thres                                                         
0.3            357.946747  370.223772  76.0  1932.787217  46.117541   
0.5            354.948544  368.629108  85.0  1910.295134  38.268083   
0.7            355.718552  364.833884  96.0  1913.727728  32.263417   

                                                                   \
                       min          25%          50%          75%   
reg_xgb_thres                                                       
0.3            1848.787198  1905.566473  1929.491158  1958.481577   
0.5            1847.980734  1878.533794  1912.114413  1930.007720   
0.7            1841.159497  1889.400818  1908.104529  1936.916978   

                            mape                                       \
                       max count         mean         std         min   
reg_xgb_thres                                                           
0.3            2044.116309  76.0  1633.481192  483.369407  890.663711   
0.5            2010.124756  85.0  1448.490730  295.610353  930.227428   
0.7            2000.766719  96.0  1482.221074  211.694518  887.501279   

                                                                  precision  \
                       25%          50%          75%          max     count   
reg_xgb_thres                                                                 
0.3            1366.246098  1463.211887  1781.211416  3076.204072      76.0   
0.5            1276.749608  1422.809316  1559.604577  2474.692942      85.0   
0.7            1333.232396  1473.248129  1594.527955  2019.687518      96.0   

                                                                           \
                   mean       std       min       25%       50%       75%   
reg_xgb_thres                                                               
0.3            0.328789  0.009771  0.300744  0.320678  0.332224  0.335873   
0.5            0.329040  0.009708  0.298232  0.323589  0.331413  0.335623   
0.7            0.330653  0.006803  0.314039  0.328067  0.332458  0.334914   

                        recall                                          \
                    max  count      mean       std       min       25%   
reg_xgb_thres                                                            
0.3            0.343743   76.0  0.376130  0.030544  0.284227  0.363938   
0.5            0.342301   85.0  0.379569  0.026430  0.294118  0.372202   
0.7            0.342812   96.0  0.384750  0.019451  0.317803  0.378123   

                                            f1_score                      \
                    50%       75%       max    count      mean       std   
reg_xgb_thres                                                              
0.3            0.388209  0.397579  0.409162     76.0  0.350577  0.018879   
0.5            0.388600  0.396148  0.412025     85.0  0.352304  0.016870   
0.7            0.390031  0.397254  0.408381     96.0  0.355532  0.011899   

                                                                 
                    min       25%       50%       75%       max  
reg_xgb_thres                                                    
0.3            0.292252  0.343620  0.358237  0.364102  0.371046  
0.5            0.296160  0.346976  0.357824  0.363006  0.373819  
0.7            0.316076  0.351602  0.358557  0.363073  0.371977

In [16]:
combined_metrics.sort_values('f1_score', ascending=False)

,iteration,timestamp,run_id,clf_xgb_n_estimators,clf_xgb_max_depth,clf_xgb_learning_rate,clf_xgb_gamma,clf_xgb_min_child_weight,clf_xgb_reg_alpha,clf_xgb_reg_lambda,reg_xgb_n_estimators,reg_xgb_max_depth,reg_xgb_learning_rate,reg_xgb_gamma,reg_xgb_min_child_weight,reg_xgb_reg_alpha,reg_xgb_reg_lambda,lr_penalty,lr_C,enet_alpha,enet_l1_ratio,threshold_beta,clf_xgb_thres,reg_xgb_thres,mae,rmse,mape,precision,recall,f1_score,accuracy,true_positives,false_positives,true_negatives,false_negatives,total_samples,actual_binding_count,predicted_binding_count,actual_binding_rate,predicted_binding_rate,metrics_file,per_outage_file,agg_file
27,88,20251119_141329,7236,300,5,0.20,0.1,10,0.0,10,200,4,0.05,0.1,10,0.0,1,l2,1.0,10.0,0.5,0.5,0.3,0.5,349.586777,1912.571449,1496.219807,0.342276,0.411765,0.373819,0.881046,1582,3040,37673,2260,44555,3842,4622,0.086231,0.103737,/opt/temp/haoyan/param_search/metrics/iter_88_...,/opt/temp/haoyan/param_search/per_outage/iter_...,/opt/temp/haoyan/param_search/agg/iter_88_2025...
220,272,20251119_184155,4890,300,4,0.10,0.1,10,1.0,10,300,4,0.10,0.2,5,1.0,1,l2,1.0,1.0,0.9,0.1,0.5,0.5,353.594545,1925.179265,1454.961977,0.342301,0.407340,0.371999,0.881405,1565,3007,37706,2277,44555,3842,4572,0.086231,0.102615,/opt/temp/haoyan/param_search/metrics/iter_272...,/opt/temp/haoyan/param_search/per_outage/iter_...,/opt/temp/haoyan/param_search/agg/iter_272_202...
222,376,20251119_195638,3990,500,4,0.20,0.0,10,1.0,100,100,3,0.10,0.1,5,1.0,1,l2,10.0,10.0,0.1,0.5,0.5,0.7,352.492333,1924.602867,1555.500672,0.341532,0.408381,0.371977,0.881091,1569,3025,37688,2273,44555,3842,4594,0.086231,0.103109,/opt/temp/haoyan/param_search/metrics/iter_376...,/opt/temp/haoyan/param_search/per_outage/iter_...,/opt/temp/haoyan/param_search/agg/iter_376_202...
123,338,20251119_200329,5832,200,4,0.20,0.0,10,0.1,1,500,5,0.10,0.0,1,1.0,1,l2,1.0,1.0,0.5,3.0,0.5,0.3,347.887217,1879.851251,1457.556893,0.340509,0.407600,0.371046,0.880844,1566,3033,37680,2276,44555,3842,4599,0.086231,0.103221,/opt/temp/haoyan/param_search/metrics/iter_338...,/opt/temp/haoyan/param_search/per_outage/iter_...,/opt/temp/haoyan/param_search/agg/iter_338_202...
133,378,20251119_195457,9814,500,6,0.20,0.0,5,1.0,100,200,6,0.20,0.2,10,0.1,100,l2,1.0,0.1,0.9,0.5,0.3,0.3,360.753324,2009.261003,1592.851634,0.341972,0.405258,0.370935,0.881472,1557,2996,37717,2285,44555,3842,4553,0.086231,0.102188,/opt/temp/haoyan/param_search/metrics/iter_378...,/opt/temp/haoyan/param_search/per_outage/iter_...,/opt/temp/haoyan/param_search/agg/iter_378_202...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,200,20251119_175350,6077,200,4,0.05,0.2,1,0.0,100,300,5,0.20,0.2,1,0.0,1,l2,10.0,10.0,0.9,0.1,0.3,0.5,354.169797,1943.887727,1390.880759,0.306920,0.302447,0.304667,0.880956,1162,2624,38089,2680,44555,3842,3786,0.086231,0.084974,/opt/temp/haoyan/param_search/metrics/iter_200...,/opt/temp/haoyan/param_search/per_outage/iter_...,/opt/temp/haoyan/param_search/agg/iter_200_202...
148,360,20251119_194106,8981,100,6,0.05,0.2,5,0.1,100,300,4,0.20,0.2,5,1.0,100,l2,10.0,1.0,0.9,1.0,0.3,0.5,352.059289,1954.976086,1412.547391,0.311547,0.297762,0.304498,0.882707,1144,2528,38185,2698,44555,3842,3672,0.086231,0.082415,/opt/temp/haoyan/param_search/metrics/iter_360...,/opt/temp/haoyan/param_search/per_outage/iter_...,/opt/temp/haoyan/param_search/agg/iter_360_202...
77,8,20251119_132707,1545,100,3,0.05,0.2,10,1.0,10,200,3,0.05,0.1,10,0.1,1,l2,1.0,0.1,0.1,0.5,0.5,0.3,354.642744,1943.327772,1327.195224,0.303744,0.301926,0.302833,0.880126,1160,2659,38054,2682,44555,3842,3819,0.086231,0.085714,/opt/temp/haoyan/param_search/metrics/iter_8_2...,/opt/temp/haoyan/param_search/per_outage/iter_...,/opt/temp/haoyan/param_search/agg/iter_8_20251...
149,44,20251119_134426,4667,100,3,0.05,0.0,5,0.1,10,100,3,0.10,0.2,1,0.1,10,l2,0.1,1.0,0.9,3.0,0.3,0.5,350.193625,1927.990711,1850.762100,0

In [43]:
x = pd.read_parquet(combined_metrics.sort_values('f1_score', ascending=False).loc[1]['agg_file'])
cols = ['90_diff', '105_diff', '95_diff', '100_diff', 'curvature_100', '60_diff', 'prob_overload', '90_diff', '85_diff', '100_diff', '70_diff', '95_diff', 'prob_exceed_100', '80_diff', 'prob_exceed_90', '105', 'log_density_100', '110', 'risk_ratio', 'actual_shadow_price', 'predicted_shadow_price', 'binding_probability', 'predicted_binding_count']
x[x['predicted_binding_count'] >= 1][cols].corr(method='spearman')['actual_shadow_price'].sort_values(ascending=False)

actual_shadow_price        1.000000
binding_probability        0.408333
70_diff                    0.235587
60_diff                    0.234092
80_diff                    0.224915
85_diff                    0.216423
predicted_binding_count    0.213911
predicted_shadow_price     0.211007
90_diff                    0.208396
90_diff                    0.208396
95_diff                    0.198098
95_diff                    0.198098
prob_exceed_90             0.192556
log_density_100            0.191118
105                        0.190501
105_diff                   0.188552
100_diff                   0.188497
100_diff                   0.188497
110                        0.185019
prob_exceed_100            0.179159
prob_overload              0.179159
curvature_100              0.162383
risk_ratio                 0.117871
Name: actual_shadow_price, dtype: float64

In [8]:
if 'combined_metrics' in locals():
    # Display summary statistics
    print("Summary Statistics:")
    print("=" * 60)
    
    metric_cols = ['f1_score', 'precision', 'recall', 'mae', 'rmse']
    available_metrics = [col for col in metric_cols if col in combined_metrics.columns]
    
    summary = combined_metrics[available_metrics].describe()
    print(summary)
    
    # Optionally display the top 5 runs by F1
    if 'F1' in combined_metrics.columns:
        print("\n" + "=" * 60)
        print("Top 5 Runs by F1 Score")
        print("=" * 60)
        
        display_cols = ['iteration'] + available_metrics
        top5 = combined_metrics.nlargest(5, 'F1')[display_cols]
        print(top5.to_string(index=False))

Summary Statistics:
         f1_score   precision      recall         mae         rmse
count  257.000000  257.000000  257.000000  257.000000   257.000000
mean     0.352999    0.329568    0.380487  351.918539  1918.228703
std      0.015958    0.008763    0.025606    6.301354    39.765760
min      0.292252    0.298232    0.284227  338.131928  1841.159497
25%      0.348225    0.325378    0.374805  347.071433  1889.130928
50%      0.358155    0.331787    0.389120  351.526284  1915.072166
75%      0.363571    0.335542    0.397189  355.997848  1943.437691
max      0.373819    0.343743    0.412025  370.223772  2044.116309


In [ ]:
# import shutil

# shutil.rmtree(BASE_OUTPUT_DIR)